<a href="https://colab.research.google.com/github/treborskrub/Modular-/blob/main/pipeline_linker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

from typing import Dict, Any

class UPCPipelineLinker:
    """Orchestrator that glues gate + sieve + your audit engine together."""
    def __init__(self, gate_engine: Any, sieve_engine: Any) -> None:
        self.gate = gate_engine
        self.sieve = sieve_engine
        self.active_pipelines: Dict[str, Dict[str, any]] = {}

    def ingest_packet(self, session_id: str, packet: Dict[str, any]) -> Dict[str, any]:
        state = self.active_pipelines.get(session_id, {"phase": "GATE", "step": 1})

        if state["phase"] == "GATE":
            gate_status = self.gate.process_handshake_step(packet, state["step"])
            if gate_status == "CONNECTION_TERMINATED":
                self.active_pipelines.pop(session_id, None)
                return {"status": "HALTED", "reason": "Handshake Authentication Failure"}
            elif gate_status == "PIPELINE_UNLOCKED":
                state["phase"] = "SIEV"
                state["pass_count"] = 1
                self.active_pipelines[session_id] = state
                return {"status": "PIPELINE_UNLOCKED", "phase": "SIEV", "next_expected_pass": 1}
            else:
                state["step"] += 1
                self.active_pipelines[session_id] = state
                return {"status": "HANDSHAKING", "next_expected_step": state["step"]}

        elif state["phase"] == "SIEV":
            packet["base_perplexity"] = packet.get("initial_delta", 1.0)
            packet["current_token_loss"] = packet.get("current_delta", 0.05)
            sieve_result = self.sieve.run_sieve(packet, state["pass_count"])

            if sieve_result["status"] == "PROCESSING":
                state["pass_count"] += 1
                self.active_pipelines[session_id] = state
                return {"status": "PROCESSING_STREAM", "lambda": sieve_result["lambda"]}
            elif sieve_result["status"] == "VERIFIED":
                self.active_pipelines.pop(session_id, None)
                return {"status": "SUCCESS_STREAM_VERIFIED", "meta": sieve_result["meta"]}
            elif sieve_result["status"] == "DIVERGED_FALLBACK":
                self.active_pipelines.pop(session_id, None)
                return {"status": "CRITICAL_DIVERGENCE_EVACUATED"}

        return {"status": "UNKNOWN_ERROR"}